# Mode B1: Memetic + Constraint-Aware Repair Operators

**Enhancement over Mode B**: Replaces blind local search with 7 constraint-aware repair heuristics.

| Aspect | Mode B (Baseline) | Mode B1 (This) |
|--------|-------------------|----------------|
| Strategy | Random hill-climbing | Priority-ordered repair |
| Operators | 3 (time, room, instructor) | 7 (constraint-specific) |
| Awareness | Blind to constraints | Directly fixes violations |

## 1. Imports

In [7]:
from __future__ import annotations
import random, copy, time
import numpy as np
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field

from deap import base, creator, tools
from schedule_engine.notebooks.core import (
    load_data, create_random_individual, course_aware_crossover, smart_mutation,
    create_evaluator, get_constraint_breakdown, setup_deap, get_best_individual, 
    EvolutionStats, print_constraint_details
)
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary
from schedule_engine.domain.gene import SessionGene
from schedule_engine.domain.types import SchedulingContext

# MODE B1: Import production repair operators
from schedule_engine.ga.operators.repair import (
    repair_instructor_availability,
    repair_group_overlaps,
    repair_room_overlap_reassign,
    repair_room_conflicts,
    repair_instructor_conflicts,
    repair_instructor_qualifications,
    repair_room_type_mismatches,
)

print(" Imports successful (including 7 repair operators)")

 Imports successful (including 7 repair operators)


## 2. Mode B1 Configuration

In [8]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters (same as Mode B for fair comparison)
POP_SIZE = 10
NGEN = 100
CXPB = 0.9
MUTPB = 0.2
FITNESS_WEIGHTS = (-1.0, -1.0)

# MODE B1: Repair operator parameters
REPAIR_PROB = 0.2  # Same as Mode B local search probability
REPAIR_ITERATIONS = 3  # Number of repair passes

# Paths
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_b1_repair_operators/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Mode B1: pop={POP_SIZE}, ngen={NGEN}, repair_prob={REPAIR_PROB}")

 Mode B1: pop=10, ngen=100, repair_prob=0.2


## 3. Load Data

In [9]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

# Get context for repair operators
context = data.context
evaluate = create_evaluator(data)

print(f" {data.summary()}")

️  Non-schedulable courses filtered out

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (0 credits/LTP)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (0 credits/LTP)

ENIE 254: BIE4A, BIE4B (0 credits/LTP)

ME706: BME7A, BME7B (0 credits/LTP)

16 enrollments skipped (Survey Camp, Industrial Attachment, etc.)

 Courses: 668, Groups: 74, Instructors: 181, Rooms: 67, Quanta: 42


In [10]:
def create_truly_random_individual(data: "NotebookData") -> list[SessionGene]:
    """
    Create a TRULY random individual with course-group structure preserved.
    
    Preserves:
        - Course-group pairs (no pedagogical violations)
        - Number of quanta per course
    
    Random (can violate):
        - Instructor assignment (any instructor, can be unqualified)
        - Room assignment (any room, can be wrong type/size)
        - Time assignment (any quanta, can have conflicts)
    
    This creates ~4000-5000 violations vs ~1000-1500 with smart init.
    """
    from schedule_engine.ga.population import generate_course_group_pairs, analyze_group_hierarchy
    
    # Get course-group pairs (preserves pedagogical structure)
    hierarchy = analyze_group_hierarchy(data.context.groups)
    pair_tuples = generate_course_group_pairs(
        data.context.courses, 
        data.context.groups, 
        hierarchy, 
        silent=True
    )
    
    # Convert to simpler format
    course_group_pairs = [
        (course_key, group_ids, num_quanta) 
        for course_key, group_ids, _, num_quanta in pair_tuples
    ]
    
    # Get all available resources (for random selection)
    all_instructors = list(data.instructors.values())
    all_rooms = list(data.rooms.values())
    all_quanta = list(range(data.qts.total_quanta))
    
    genes = []
    for course_id, group_ids, num_quanta in course_group_pairs:
        # TRULY RANDOM: Any instructor, room, time
        instructor = random.choice(all_instructors)
        room = random.choice(all_rooms)
        
        # Random contiguous time block (start_quanta)
        max_start = len(all_quanta) - num_quanta
        if max_start > 0:
            start_quanta = random.randint(0, max_start)
        else:
            start_quanta = 0
        
        # Get course info for session type
        course = data.courses.get(course_id)
        course_type = course.course_type if course else "theory"
        
        gene = SessionGene(
            course_id=course_id[0] if isinstance(course_id, tuple) else course_id,
            course_type=course_type,
            group_ids=group_ids,
            instructor_id=instructor.instructor_id,
            room_id=room.room_id,
            start_quanta=start_quanta,
            num_quanta=num_quanta,
        )
        genes.append(gene)
    
    return genes


# Compare smart vs truly random initialization
print(" Comparing Initialization Strategies")
print("=" * 70)

smart_ind = create_random_individual(data)
smart_fitness = evaluate(smart_ind)
smart_breakdown = get_constraint_breakdown(smart_ind, data)

truly_random_ind = create_truly_random_individual(data)
random_fitness = evaluate(truly_random_ind)
random_breakdown = get_constraint_breakdown(truly_random_ind, data)

print(f"\n Smart Initialization (Constraint-Guided):")
print(f"   Hard: {smart_fitness[0]:.0f}, Soft: {smart_fitness[1]:.0f}")
print(f"   Top violations: ", end="")
hard_names = {'student_group_exclusivity', 'instructor_exclusivity', 'instructor_qualifications', 
             'room_suitability', 'room_exclusivity', 'instructor_time_availability', 
             'room_time_availability', 'course_completeness'}
top_smart = sorted([(k, v) for k, v in smart_breakdown.items() if k in hard_names and v > 0], 
                   key=lambda x: x[1], reverse=True)[:3]
print(", ".join(f"{k}={v:.0f}" for k, v in top_smart))

print(f"\n Truly Random Initialization:")
print(f"   Hard: {random_fitness[0]:.0f}, Soft: {random_fitness[1]:.0f}")
print(f"   Top violations: ", end="")
top_random = sorted([(k, v) for k, v in random_breakdown.items() if k in hard_names and v > 0], 
                    key=lambda x: x[1], reverse=True)[:3]
print(", ".join(f"{k}={v:.0f}" for k, v in top_random))

print(f"\n For ablation study, use TRULY RANDOM to show repair effectiveness!")
print("=" * 70)

 Comparing Initialization Strategies

 Smart Initialization (Constraint-Guided):
   Hard: 2495, Soft: 1332
   Top violations: student_group_exclusivity=792, instructor_qualifications=516, instructor_time_availability=413

 Truly Random Initialization:
   Hard: 2632, Soft: 1362
   Top violations: student_group_exclusivity=872, instructor_qualifications=518, instructor_time_availability=401

 For ablation study, use TRULY RANDOM to show repair effectiveness!


## 3a. Create Truly Random Individual (No Smart Initialization)

**Key Difference from `create_random_individual()`:**

| Aspect | Smart Init (Current) | Truly Random (New) |
|--------|---------------------|-------------------|
| Course-Group Pairs | ✅ Preserved | ✅ Preserved |
| Instructor Assignment | Qualified only | **Any instructor** |
| Room Assignment | Suitable only | **Any room** |
| Time Assignment | Conflict-free | **Any quanta** |
| Expected Violations | ~1000-1500 | ~4000-5000 |

## 4. Repair Operator Wrapper

**Priority Order** (lower = first):
1. `repair_instructor_availability` - Fix part-time instructor time violations
2. `repair_group_overlaps` - Fix student group double-booking
3. `repair_room_overlap_reassign` - Move to idle rooms
4. `repair_room_conflicts` - Shift times if no room available
5. `repair_instructor_conflicts` - Fix instructor double-booking
6. `repair_instructor_qualifications` - Reassign to qualified instructors
7. `repair_room_type_mismatches` - Match room type to course needs

In [11]:
@dataclass
class RepairStats:
    """Track repair operator statistics."""
    total_fixes: int = 0
    by_operator: dict[str, int] = field(default_factory=dict)


def apply_repair_operators(
    individual: list[SessionGene],
    context: SchedulingContext,
    max_iterations: int = 3,
) -> RepairStats:
    """
    Apply constraint-aware repair operators in priority order.
    
    Args:
        individual: Individual to repair (modified in-place)
        context: Scheduling context
        max_iterations: Number of repair passes
        
    Returns:
        RepairStats with fix counts
    """
    stats = RepairStats()
    
    repair_operators = [
        ("instructor_availability", repair_instructor_availability),
        ("group_overlaps", repair_group_overlaps),
        ("room_overlap_reassign", repair_room_overlap_reassign),
        ("room_conflicts", repair_room_conflicts),
        ("instructor_conflicts", repair_instructor_conflicts),
        ("instructor_qualifications", repair_instructor_qualifications),
        ("room_type_mismatches", repair_room_type_mismatches),
    ]
    
    for _ in range(max_iterations):
        iteration_fixes = 0
        for name, operator in repair_operators:
            try:
                fixes = operator(individual, context)
                if fixes > 0:
                    stats.by_operator[name] = stats.by_operator.get(name, 0) + fixes
                    stats.total_fixes += fixes
                    iteration_fixes += fixes
            except Exception:
                pass  # Skip failed operators
        
        if iteration_fixes == 0:
            break  # No more fixes possible
    
    return stats


# Test repair operators on TRULY RANDOM individual
print(" Testing repair operators on truly random individual")
test_ind = create_truly_random_individual(data)
fitness_before = evaluate(test_ind)
repair_stats = apply_repair_operators(test_ind, context, max_iterations=3)
fitness_after = evaluate(test_ind)

print(f"   Before: Hard={fitness_before[0]:.0f}, Soft={fitness_before[1]:.0f}")
print(f"   After:  Hard={fitness_after[0]:.0f}, Soft={fitness_after[1]:.0f}")
print(f"   Reduction: {fitness_before[0] - fitness_after[0]:.0f} hard violations fixed")
print(f"\n   Total fixes applied: {repair_stats.total_fixes}")
if repair_stats.by_operator:
    print("   Breakdown:")
    for op, count in sorted(repair_stats.by_operator.items(), key=lambda x: x[1], reverse=True):
        print(f"      - {op}: {count}")

 Testing repair operators on truly random individual
   Before: Hard=2470, Soft=1322
   After:  Hard=973, Soft=1198
   Reduction: 1497 hard violations fixed

   Total fixes applied: 3846
   Breakdown:
      - room_conflicts: 1527
      - instructor_conflicts: 1309
      - room_overlap_reassign: 559
      - instructor_availability: 161
      - instructor_qualifications: 149
      - group_overlaps: 134
      - room_type_mismatches: 7


## 5. Memetic NSGA-II with Repair Operators (Mode B1)

In [ ]:
LOG_INTERVAL = 10

def run_memetic_repair_nsga2():
    """Run NSGA-II with constraint-aware repair operators (TRULY RANDOM INIT)."""
    print(f" Mode B1: pop={POP_SIZE}, ngen={NGEN}, repair_prob={REPAIR_PROB}")
    print(f"   Using TRULY RANDOM initialization (not smart)")
    start = time.time()
    
    setup_deap(FITNESS_WEIGHTS)
    
    toolbox = base.Toolbox()
    # CHANGE: Use truly random instead of smart initialization
    toolbox.register("individual", lambda: creator.Individual(create_truly_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    total_repairs = 0
    
    for gen in range(NGEN):
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # === MODE B1: Constraint-Aware Repair ===
        for ind in offspring:
            if random.random() < REPAIR_PROB:
                repair_stats = apply_repair_operators(list(ind), context, REPAIR_ITERATIONS)
                total_repairs += repair_stats.total_fixes
                del ind.fitness.values
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        
        if gen % LOG_INTERVAL == 0 or gen == NGEN - 1:
            best_ind = min(pop, key=lambda ind: (ind.fitness.values[0], ind.fitness.values[1]))
            breakdown = get_constraint_breakdown(list(best_ind), data)
            hard_names = {'student_group_exclusivity', 'instructor_exclusivity', 'instructor_qualifications', 
                         'room_suitability', 'room_exclusivity', 'instructor_time_availability', 
                         'room_time_availability', 'course_completeness'}
            hard_bd = {k: v for k, v in breakdown.items() if k in hard_names}
            soft_bd = {k: v for k, v in breakdown.items() if k not in hard_names}
            print_constraint_details(hard_bd, soft_bd, gen)
    
    stats.elapsed_time = time.time() - start
    print(f" Done in {stats.elapsed_time:.1f}s (total repairs: {total_repairs})")
    return pop, stats

final_pop, stats = run_memetic_repair_nsga2()

 Mode B1: pop=10, ngen=100, repair_prob=0.2
   Using TRULY RANDOM initialization (not smart)
  Gen   0:  Hard=2318  Soft=1324
         HARD: [course_comple=   0 | instructor_ex= 124 | instructor_qu= 404 | instructor_ti= 407 | room_exclusiv= 369 | room_suitabil= 233 | room_time_ava=   0 | student_group= 781]
         SOFT: [instructor_sc=  15 | paired_cohort=   0 | session_conti= 875 | student_lunch= 378 | student_sched=  56]


## 5a. DEBUG: Check Initial Population Quality

**IMPORTANT**: `create_random_individual()` uses constraint-guided smart initialization, NOT truly random!

This explains why Gen 0 has ~1024 hard violations instead of ~4000:
- Uses `create_session_gene_with_conflict_avoidance()`
- Finds qualified instructors
- Avoids time conflicts
- Finds suitable rooms

To compare with truly random initialization, we'd need to bypass smart initialization.

In [ ]:
# Create a sample initial individual to show starting quality
print(" DEBUG: Initial Individual Quality")
print("=" * 60)

debug_ind = create_random_individual(data)
debug_fitness = evaluate(debug_ind)
debug_breakdown = get_constraint_breakdown(debug_ind, data)

print(f"\n Gen 0 Individual Fitness:")
print(f"   Hard Violations: {debug_fitness[0]:.0f}")
print(f"   Soft Penalty: {debug_fitness[1]:.0f}")

print(f"\n Constraint Breakdown:")
hard_names = {'student_group_exclusivity', 'instructor_exclusivity', 'instructor_qualifications', 
             'room_suitability', 'room_exclusivity', 'instructor_time_availability', 
             'room_time_availability', 'course_completeness'}

print("\n   HARD Constraints:")
for name, value in sorted(debug_breakdown.items()):
    if name in hard_names and value > 0:
        print(f"      {name:35s}: {value:6.0f}")

print("\n   SOFT Constraints:")
for name, value in sorted(debug_breakdown.items()):
    if name not in hard_names and value > 0:
        print(f"      {name:35s}: {value:6.0f}")

print(f"\n This shows smart initialization creates semi-feasible individuals")
print(f"   (NOT truly random - uses constraint-guided placement)")
print("=" * 60)

## 6. Results & Visualization

In [ ]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_b1_convergence.png", title_prefix="Mode B1: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_b1_breakdown.png", title="Mode B1: Constraint Violations")

## 7. Export Results

In [ ]:
from schedule_engine.notebooks.export import export_full_results

export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_b1_repair_operators",
)

print(f"\n All files saved to: {OUTPUT_DIR}")